# Notebook 01 — Datenbank-Speicherung

**SBB Tracker · ZHAW Scientific Programming FS2026**
Joël Hasler & Patrick Ferreira

In diesem ersten Notebook werden die rohen Datensätze (Bahnhöfe,
Wetterdaten, Ist-Daten = Soll-/Ist-Vergleich der SBB) in eine
**SQLite-Datenbank** überführt. Diese DB dient als zentraler
Datenspeicher für die nachfolgenden Notebooks und ermöglicht
SQL-Queries als Bonus-Kriterium (Datenbank + SQL).

## Datenquellen

1. **Stationen** — `data/raw/stations.csv` (1'743 aktive Schweizer Bahnhöfe
   mit Geo-Koordinaten, SBB Open Data)
2. **Wetter** — `data/raw/weather/*_hourly_recent.parquet` (15 MeteoSchweiz-
   Stationen nahe der wichtigsten Bahnhöfe, stündlich)
3. **Ist-Daten** — `data/raw/istdaten/*_sbb.parquet` (gefilterte SBB-Zug-
   Events: geplante vs. tatsächliche An-/Abfahrtszeiten, 50 Tage)

Die Roh-Downloads selbst werden in `scripts/download_*.py` durchgeführt
und sind als reine Python-Scripts implementiert (Web-Scraper / API-Calls
als zweites Bonus-Kriterium).


## Bibliotheken und Einstellungen


In [1]:
# Bibliotheken und Einstellungen
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(color_codes=True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Aktuelles Verzeichnis:", os.getcwd())
print("Projekt-Root:", PROJECT_ROOT)


Aktuelles Verzeichnis: C:\Users\hasle\Documents\Scientific Programming\SBB Tracker\project\notebooks
Projekt-Root: C:\Users\hasle\Documents\Scientific Programming\SBB Tracker\project


## Datenbank initialisieren

Wir nutzen `sqlite3` direkt (keine ORM-Schicht wie SQLAlchemy), gemäss
Dozenten-Konvention. Die DB liegt unter `data/processed/sbb_tracker.db`
und enthält drei Tabellen:

- `stations` — Stamm­daten der Bahnhöfe (BPUIC, Name, lat/lon, Kanton)
- `weather_hourly` — Stündliche Wetterdaten pro MeteoSchweiz-Station
- `delays` — Verspätungs-Events pro Zug-Halt mit `delay_arr_sec` /
  `delay_dep_sec`

Existierende Tabellen werden überschrieben (`if_exists='replace'`),
damit das Notebook idempotent ist.


In [2]:
DB_PATH = DATA_PROCESSED / "sbb_tracker.db"
if DB_PATH.exists():
    DB_PATH.unlink()
print(f"DB-Pfad: {DB_PATH}")


DB-Pfad: C:\Users\hasle\Documents\Scientific Programming\SBB Tracker\project\data\processed\sbb_tracker.db


## Tabelle 1: Bahnhof-Stammdaten

Die Stations-CSV wurde von `data.sbb.ch` heruntergeladen
(Web-API → CSV-Export). Wir filtern auf aktive Bahn-Stationen mit
Geo-Koordinaten und speichern die wichtigsten Spalten.


In [3]:
df_stations = pd.read_csv(DATA_RAW / "stations.csv", sep=";")
df_stations.columns = df_stations.columns.str.lower()
print(f"Geladen: {len(df_stations):,} Bahnhoefe, {df_stations.shape[1]} Spalten")
df_stations.head(3)


Geladen: 1,743 Bahnhoefe, 9 Spalten


,number,sloid,designationofficial,abbreviation,cantonabbreviation,cantonname,lat,lon,meansoftransport
0,8517485,ch:1:sloid:17485,Sur-le-Buis,SLB,VD,Vaud,46.353300,7.100630,TRAIN
1,8517484,ch:1:sloid:17484,Aigle-Parc Aventure,AGHO,VD,Vaud,46.315300,6.986220,TRAIN
2,8518243,ch:1:sloid:18243,Beringerfeld,BERD,SH,Schaffhausen,47.695372,8.592182,TRAIN


In [4]:
conn = sqlite3.connect(DB_PATH)
df_stations.to_sql("stations", conn, if_exists="replace", index=False)
# Verifikation
check = pd.read_sql("SELECT COUNT(*) AS n FROM stations", conn)
print(f"In DB gespeichert: {check.iloc[0]['n']:,} Stationen")
conn.close()


In DB gespeichert: 1,743 Stationen


## Tabelle 2: Wetterdaten

15 MeteoSchweiz-Stationen (SMA Zürich, BER Bern, GVE Genève, BAS Basel,
LUG Lugano, …) wurden mit stündlicher Auflösung heruntergeladen. Jede
Station ist ein separates Parquet — wir concat'en sie und ergänzen
eine `station_abbr`-Spalte.


In [5]:
weather_files = sorted((DATA_RAW / "weather").glob("*_hourly_recent.parquet"))
print(f"Wetter-Parquets: {len(weather_files)}")
for f in weather_files[:3]:
    print(" ", f.name)
print(" ...")


Wetter-Parquets: 15
  BAS_hourly_recent.parquet
  BER_hourly_recent.parquet
  CHU_hourly_recent.parquet
 ...


In [6]:
weather_dfs = []
for f in weather_files:
    abbr = f.stem.split("_")[0]
    df = pd.read_parquet(f)
    df["station_abbr"] = abbr
    weather_dfs.append(df)
df_weather = pd.concat(weather_dfs, ignore_index=True)
df_weather.columns = df_weather.columns.str.lower()
# Wichtigste Spalten herausgreifen
keep_cols = ["station_abbr", "reference_timestamp",
             "tre200h0",   # Temperatur 2m (°C)
             "rre150h0",   # Niederschlag Stundensumme (mm)
             "fkl010h0",   # Windgeschwindigkeit (m/s)
             "ure200h0",   # Relative Feuchtigkeit (%)
             "sre000h0"]   # Sonnenscheindauer (Min)
keep_cols = [c for c in keep_cols if c in df_weather.columns]
df_weather_slim = df_weather[keep_cols].copy()
print(f"Wetterdaten gesamt: {len(df_weather_slim):,} Zeilen, "
      f"{df_weather_slim['station_abbr'].nunique()} Stationen")


Wetterdaten gesamt: 50,040 Zeilen, 15 Stationen

In [7]:
conn = sqlite3.connect(DB_PATH)
df_weather_slim.to_sql("weather_hourly", conn, if_exists="replace", index=False)
check = pd.read_sql("SELECT COUNT(*) AS n FROM weather_hourly", conn)
print(f"In DB gespeichert: {check.iloc[0]['n']:,} Wetter-Records")
conn.close()


In DB gespeichert: 50,040 Wetter-Records

## Tabelle 3: Verspätungsdaten (Ist-Daten)

Pro Tag eine Parquet-Datei mit den SBB-Zug-Events. Die Daten wurden
bereits beim Download auf `PRODUKT_ID == 'Zug'` und `BETREIBER_ABK == 'SBB'`
gefiltert (statt 2.5 Mio Roh-Events pro Tag bleiben ~65k SBB-relevante).

Wir speichern alle Tage zusammen, behalten aber nur die für die Analyse
wesentlichen Spalten, um die DB-Grösse moderat zu halten.


In [8]:
delay_files = sorted((DATA_RAW / "istdaten").glob("*_sbb.parquet"))
print(f"Tages-Parquets: {len(delay_files)}")
print(f"Zeitraum: {delay_files[0].stem[:10]}  bis  {delay_files[-1].stem[:10]}")


Tages-Parquets: 48
Zeitraum: 2026-03-31  bis  2026-05-19


In [9]:
delay_cols = [
    "betriebstag", "fahrt_bezeichner", "linien_text", "verkehrsmittel_text",
    "bpuic", "haltestellen_name", "sloid",
    "ankunftszeit", "an_prognose", "an_prognose_status",
    "abfahrtszeit", "ab_prognose", "ab_prognose_status",
    "delay_arr_sec", "delay_dep_sec",
    "faellt_aus_tf", "durchfahrt_tf",
]
delay_dfs = []
for f in delay_files:
    df = pd.read_parquet(f, columns=[c for c in delay_cols if True])
    df_slim = df[[c for c in delay_cols if c in df.columns]].copy()
    delay_dfs.append(df_slim)
df_delays = pd.concat(delay_dfs, ignore_index=True)
print(f"Total Records: {len(df_delays):,}  ({len(df_delays)/1e6:.2f} Mio)")


Total Records: 3,200,604  (3.20 Mio)


In [10]:
conn = sqlite3.connect(DB_PATH)
df_delays.to_sql("delays", conn, if_exists="replace", index=False)
# Indizes auf Join-Keys fuer schnellere Queries
conn.execute("CREATE INDEX IF NOT EXISTS idx_delays_bpuic ON delays(bpuic)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_delays_betriebstag ON delays(betriebstag)")
conn.commit()
check = pd.read_sql("SELECT COUNT(*) AS n FROM delays", conn)
print(f"In DB gespeichert: {check.iloc[0]['n']:,} Verspaetungs-Records")
print(f"DB-Groesse: {DB_PATH.stat().st_size / 1024**2:.1f} MB")
conn.close()


In DB gespeichert: 3,200,604 Verspaetungs-Records
DB-Groesse: 701.7 MB


## SQL-Queries: Erste Erkundung

Nun demonstrieren wir die wichtigsten SQL-Patterns auf der Datenbank.
Die folgenden Queries decken `SELECT`, `WHERE`, `GROUP BY`, `JOIN`,
`ORDER BY` und `LIMIT` ab — exakt die Spannweite, die für das
DB+SQL-Bonuskriterium erwartet wird.


### Query 1 — Schema-Überblick


In [11]:
conn = sqlite3.connect(DB_PATH)
schema = pd.read_sql(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'index') "
    "ORDER BY type, name", conn)
print(schema.to_string(index=False))
conn.close()


                  name  type
idx_delays_betriebstag index
      idx_delays_bpuic index
                delays table
              stations table
        weather_hourly table


### Query 2 — Verspätungs-Hotspots: Top-10 Bahnhöfe nach durchschnittlicher Ankunftsverspätung


In [12]:
conn = sqlite3.connect(DB_PATH)
q = '''
    SELECT haltestellen_name,
           COUNT(*) AS n_halte,
           ROUND(AVG(delay_arr_sec), 1) AS avg_delay_sec,
           ROUND(AVG(delay_arr_sec) / 60.0, 2) AS avg_delay_min
      FROM delays
     WHERE an_prognose_status = 'REAL'
       AND delay_arr_sec IS NOT NULL
     GROUP BY haltestellen_name
    HAVING COUNT(*) >= 1000
     ORDER BY avg_delay_sec DESC
     LIMIT 10
'''
top_late = pd.read_sql(q, conn)
conn.close()
top_late


,haltestellen_name,n_halte,avg_delay_sec,avg_delay_min
0,Buchs SG,2405,195.5,3.26
1,St. Margrethen SG,2311,169.2,2.82
2,Basel St. Johann,1523,127.2,2.12
3,Hünenberg Zythus,3111,115.8,1.93
4,Stabio,3040,110.4,1.84
5,Neuhausen Rheinfall,2568,107.8,1.80
6,Zug Chollermüli,6232,107.4,1.79
7,Cham Alpenblick,6234,105.1,1.75
8,Zug Schutzengel,6053,103.9,1.73
9,Paradiso,7522,101.7,1.69


### Query 3 — Verspätungs-Verteilung nach Wochentag (für späteren t-Test)


In [13]:
conn = sqlite3.connect(DB_PATH)
q = '''
    SELECT strftime('%w', betriebstag) AS weekday_num,
           COUNT(*) AS n_events,
           ROUND(AVG(delay_arr_sec), 1) AS avg_delay_sec,
           ROUND(100.0 * SUM(CASE WHEN delay_arr_sec > 180 THEN 1 ELSE 0 END)
                 / COUNT(*), 2) AS pct_late_over_3min
      FROM delays
     WHERE an_prognose_status = 'REAL'
       AND delay_arr_sec IS NOT NULL
     GROUP BY weekday_num
     ORDER BY weekday_num
'''
by_dow = pd.read_sql(q, conn)
# 0=Sonntag in SQLite, mappen auf deutsche Namen
dow_names = {"0": "So", "1": "Mo", "2": "Di", "3": "Mi", "4": "Do", "5": "Fr", "6": "Sa"}
by_dow["weekday"] = by_dow["weekday_num"].map(dow_names)
conn.close()
by_dow[["weekday", "n_events", "avg_delay_sec", "pct_late_over_3min"]]


,weekday,n_events,avg_delay_sec,pct_late_over_3min
0,So,384017,33.7,3.44
1,Mo,401724,53.3,6.30
2,Di,404904,52.1,6.04
3,Mi,346851,49.0,5.36
4,Do,401325,48.3,5.14
5,Fr,405605,46.5,5.19
6,Sa,395378,34.9,3.68


### Query 4 — JOIN von Verspätungen mit Stationen-Stammdaten (zeigt Kanton + Geo-Koords)


In [14]:
conn = sqlite3.connect(DB_PATH)
q = '''
    SELECT s.cantonabbreviation AS kanton,
           COUNT(*) AS n_events,
           ROUND(AVG(d.delay_arr_sec), 1) AS avg_delay_sec
      FROM delays d
      JOIN stations s ON d.bpuic = s.number
     WHERE d.an_prognose_status = 'REAL'
       AND d.delay_arr_sec IS NOT NULL
     GROUP BY s.cantonabbreviation
    HAVING n_events >= 500
     ORDER BY avg_delay_sec DESC
'''
by_canton = pd.read_sql(q, conn)
conn.close()
by_canton


,kanton,n_events,avg_delay_sec
0,NaN,6122,122.0
1,BS,23432,70.0
2,TI,160409,69.3
3,ZG,96109,68.4
4,UR,7648,66.3
5,SH,17213,58.9
6,LU,129880,51.8
7,FR,59622,51.4
8,SZ,58244,49.6
9,ZH,780498,47.5


### Query 5 — Liniencodes mit den meisten Halten (Top-10 SBB-S-Bahn-/Fernverkehr-Linien)


In [15]:
conn = sqlite3.connect(DB_PATH)
q = '''
    SELECT linien_text, COUNT(*) AS n_halte
      FROM delays
     WHERE linien_text IS NOT NULL
     GROUP BY linien_text
     ORDER BY n_halte DESC
     LIMIT 10
'''
top_lines = pd.read_sql(q, conn)
conn.close()
top_lines


,linien_text,n_halte
0,S9,178425
1,S1,167314
2,S3,145268
3,S20,127762
4,S24,91406
5,S2,88252
6,S12,81129
7,S5,80916
8,S6,79088
9,RL4,78191


## Zusammenfassung

Drei Tabellen sind erstellt, indiziert und mit SQL erkundet:
- **`stations`** — Stamm­daten für JOIN per BPUIC
- **`weather_hourly`** — Stündliche Wetter-Messpunkte für Wetter-Korrelation
- **`delays`** — Verspätungs-Events als Hauptanalyse-Datensatz

Die nächsten Notebooks setzen darauf auf:
- `02_datenaufbereitung.ipynb` — Cleaning, Merges, Feature-Engineering
- `03_analyse_visualisierung.ipynb` — 4 Statistik-Tests + Visualisierungen


In [16]:
# System-Info (Reproduzierbarkeits-Footer)
import platform
from platform import python_version
from datetime import datetime

print("-----------------------------------")
print(os.name.upper())
print(platform.system(), "|", platform.release())
print("Datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python Version:", python_version())
print("-----------------------------------")


-----------------------------------
NT
Windows | 11
Datetime: 2026-05-20 23:46:46
Python Version: 3.12.10
-----------------------------------
